# **🤗 HUGGING FACE MODELS**


Learn to navigate **HuggingFace** (https://huggingface.co/), deploy production-ready models for free, and build a live Streamlit application that showcases your work to the world.

Let's use Collab A100 GPU

**WHY IS IMPORTANT?:**

 * We have immediate, **free access to over 500,000 AI models**, from text generation to image recognition, all deployable in minutes without infrastructure. We just need to understand which model to use for which task.

 * Every model uses the same **`pipeline()`** function with just two changes: the task type and the model name. This consistency is HuggingFace's genius

 * Once we understand this pattern, you can deploy any thousands of models with minimal code changes. We are not learning 10 different APIs. But learning one interface that unlocks everything

##**Step 1:** Set Up Google Colab With Your HuggingFace Token

In [ ]:
!pip install transformers torch sentencepiece accelerate

In [ ]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

##**Step 2:** Load and Test 10 Different Models
Now comes the exciting part. You’re about to interact with 10 distinct AI models, each solving a different problem. Watch how the same code pattern works across radically different tasks.

### **Model 1: Sentiment Analysis**

In [ ]:
from transformers import pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
result = sentiment_analyzer("This has been an extremelly hard year!")
print(f"Sentiment: {result[0]['label']}, Confidence: {result[0]['score']:.2%}")

* ***Device set to use cuda: 0*** -----> Means model is running on the **GPU**, which makes it faster.

###**Model 2: Zero-Shot Classification**
Hugging Face Zero-Shot Classification **predicts the best label for text without prior training** on that label.

* You give the model a **piece of text**
* You also give it **possible labels** (that you choose)
* The model decides **which label fits best**, even if it has **never seen those labels before**

**Example:**
* Text: *“I love this movie, it was amazing!”*
* Labels: "positive", "negative", "neutral"
* Output: **positive**

**Why it’s useful:**
* No training needed
* Flexible (you can change labels anytime)
* Great for quick text analysis

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
text = "I need to cancel my subscription and get a refund."
candidate_labels = ["billing issue", "technical problem", "account management", "product inquiry"]
result = classifier(text, candidate_labels)
print(f"Category: {result['labels'][0]} (Confidence: {result['scores'][0]:.2%})")


### What's Happening:

* Small config files download.
* Large model file (1.6GB) downloads.
* Model weights are loaded into RAM.
* Tokenizer files load.
* Model becomes ready for classifier to run.

1️⃣ **config.json →** A small settings file that tells the model how it's structured.

2️⃣ **model.safetensors:** 88% 1.44G/1.63G → This is the main model file. This is the heavy part because it contains the model's learned weights.

3️⃣ **Loading weights:** loading the model into memory. It's converting stored data into usable model parameters.

4️⃣ **tokenizer_config.json:** Configuration for the tokenizer for text processing

5️⃣ **vocab.json:** Vocabulary file.

6️⃣ **merges.txt:** Used in subword tokenization



>#### **Medical text classification:**
Let's try with  5 clinical specialties

In [ ]:
from transformers import pipeline

# Load zero-shot classifier
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Example medical abstract text
text = (
    "This study aims to assess  the effectiveness of beta-blockers in reducing "
    "blood pressure and improving outcomes in patients with chronic heart failure."
)

# Candidate medical specialties
candidate_labels = [
    "cardiology",
    "neurology",
    "oncology",
    "endocrinology",
    "pulmonology"
]

# Run zero-shot classification
result = classifier(text, candidate_labels)

# Display top prediction
print(
    f"Medical Specialty: {result['labels'][0]} "
    f"(Confidence: {result['scores'][0]:.2%})"
)


### **Model 3: NER (Named Entity Recognition)**

This code uses a **Named Entity Recognition (NER)** model to **find & label important entities in text**.

* Loads a pretrained NER model (`bert-base-NER`)
* Analyzes the sentence
* Identifies entities like **organizations, people, and locations**
* Prints each entity with its **type** and **confidence score**

**In one line:**

> The code detects and labels real-world entities (people, places, organizations) in a sentence using a pretrained NLP model.


In [ ]:
from transformers import pipeline

try:
    ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
except TypeError:
    ner = pipeline("ner", model="dslim/bert-base-NER", grouped_entities=True)

text = (
    "The patient was diagnosed with diabetes and prescribed metformin by "
    "Dr. Smith at Johns Hopkins Hospital in Baltimore."
)

for e in ner(text):
    print(f"{e['word']}: {e.get('entity_group', e.get('entity'))} (Confidence: {e['score']:.2%})")



* **Smith: PER (99.42%)**
  The model identified **“Smith” as a person** with high confidence.

* **Johns Hopkins Hospital: LOC (99.93%)**
  The model identified the hospital as a **location**.

* **Baltimore: LOC (99.86%)**
  The model identified **Baltimore as a location**.

**In one line:**

> The model found people and locations in the text and is very confident about its predictions.


In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",  # replaces grouped_entities=True
)

text = "Apple Inc. announced that Tim Cook will visit Paris next Tuesday to meet with President Macron."
entities = ner(text)

for ent in entities:
    print(f"{ent['word']}: {ent['entity_group']} (Confidence: {ent['score']:.2%})")


#### **NOTE:**
New transformers version **no longer accepts the grouped_entities argument in pipeline("ner", ...)**. Use **`aggregation_strategy="simple"`** instead, which performs the same “group tokens into full entities” behavior.

###**Model 4: Question Answering**

In [ ]:
qa = pipeline("question-answering", model="deepset/roberta-base-squad2")
context = "HuggingFace is a company founded in 2016 that develops tools for building applications using machine learning. It is headquartered in New York City."
question = "When was HuggingFace founded?"
result = qa(question=question, context=context)
print(f"Answer: {result['answer']} (Confidence: {result['score']:.2%})")

###**Model 5: Text Generation - GPT-2 Model**

> Uses **GPT-2** to generate a short continuation of a given text prompt.
* Loads a **text generation model (GPT-2)**.
* Provides a **starting prompt** about AI in healthcare.
* The model **automatically continues the text** based on that prompt.
* **`temperature=0.7`** controls creativity (moderate randomness).
* Prints the **generated sentence**.

**NOTE:** Because it uses GPT-2 is only updated up to 2019


In [ ]:
from transformers import pipeline, set_seed

generator = pipeline("text-generation", model="gpt2")

set_seed(42)  # make results repeatable across runs (same env + versions)

prompt = "The future of artificial intelligence in healthcare will"
out = generator(
    prompt,
    max_new_tokens=50,          # preferred over max_length
    do_sample=True,             # explicit sampling
    temperature=0.7,
    num_return_sequences=1,
    pad_token_id=generator.tokenizer.eos_token_id,  # avoids pad/eos warnings
)

print(out[0]["generated_text"])


#### **NOTE:**
* Prefer **`max_new_tokens`** (not max_length) to avoid length-related warnings in many setups.
* Add a simple post-filter that rejects outputs containing URLs/emails (or strip them) before printing.



In [ ]:
generator = pipeline("text-generation", model="gpt2")
prompt = "Write a short, high-level research summary about Challenges in the Management of Diabetes Mellitus."
result = generator(prompt, max_length=50, num_return_sequences=1, temperature=0.7)
print(result[0]['generated_text'])

#### What each message means
* **Truncation was not explicitly activated…**
  You set a maximum length, but didn’t tell the tokenizer **how to cut long input text**.
  Hugging Face automatically truncated it for you.

* **Setting `pad_token_id` to `eos_token_id`**
  The model doesn’t have a padding token, so it uses the **end-of-sentence token** instead.
  This is normal for GPT-style models.

* **Both `max_new_tokens` and `max_length` were set**
  You gave **two length limits**.

  * `max_new_tokens` controls **how much new text is generated**
  * `max_length` controls **total length (input + output)**
    Hugging Face uses **`max_new_tokens`** and ignores `max_length`.
---
#### Why the generated text looks fake???

* The model generated **fake academic-style citations**
* Dates, journals, and authors are **hallucinated**
* This happens because:

  * The model is **not a medical specialist**
  * Text generation models **predict likely text**, not factual references

####**How to interpret this output (important):**
✅ Useful for demonstrating text generation
❌ NOT suitable for medical accuracy

###**Model 6: Translation | Seq2SeqLM, Marian (OPUS‑MT) seq2seq model**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-es"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

english_text = "HuggingFace makes artificial intelligence accessible to everyone."

inputs = tokenizer([english_text], return_tensors="pt", padding=True, truncation=True)
outputs = model.generate(**inputs, max_new_tokens=64)

spanish = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(f"Spanish: {spanish}")



`pipeline()` translation tasks are currently broken (we’re seeing KeyError: 'translation'), so the safest update is to avoid the translation pipeline entirely and run the **Marian (OPUS‑MT) seq2seq model** via generate(). This bypasses the SUPPORTED_TASKS["translation"] lookup that’s failing in your traceback.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# English -> French OPUS-MT (Marian)
model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

english_text = "HuggingFace makes artificial intelligence accessible to everyone."

inputs = tokenizer(english_text, return_tensors="pt", truncation=True)
outputs = model.generate(**inputs, max_new_tokens=64)

french = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"French: {french}")


###**Model 7: Summarization using BART Large CNN**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

long_text = """
Artificial intelligence is rapidly transforming industries across the globe. From healthcare to finance, companies are leveraging
machine learning models to automate processes, gain insights from data, and create innovative products. However, the deployment of AI
systems requires significant expertise and infrastructure, which has historically limited access to large organizations with substantial
resources. Platforms like HuggingFace are democratizing AI by providing free access to thousands of pre-trained models, eliminating the
need for expensive training processes and specialized hardware. This shift is enabling small startups and individual developers to build
sophisticated AI applications that were previously impossible without major funding.
"""

inputs = tokenizer(long_text, return_tensors="pt", truncation=True, max_length=1024)
summary_ids = model.generate(
    **inputs,
    max_new_tokens=60,      # controls summary length
    min_new_tokens=25,
    num_beams=4,            # more stable than sampling for summaries
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"Summary: {summary}")


###**Model 8: Image Classification VIT**



In [ ]:
from PIL import Image
import requests
image_classifier = pipeline("image-classification", model="google/vit-base-patch16-224")
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png"
image = Image.open(requests.get(image_url, stream=True).raw)
results = image_classifier(image)
print("Top 3 predictions:")
for i, result in enumerate(results[:3], 1):
    print(f"{i}. {result['label']}: {result['score']:.2%}")

###**Model 9: Object Detection - FaceBook ResNET 50**

In [ ]:
object_detector = pipeline("object-detection", model="facebook/detr-resnet-50")
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(image_url, stream=True).raw)
results = object_detector(image)
print("Detected objects:")
for detection in results:
    print(f"- {detection['label']}: {detection['score']:.2%} at location {detection['box']}")

#### **Run the logs quitely, keep progress bar:**

In [ ]:
from transformers import pipeline, logging
import os
import warnings
import requests
from PIL import Image

# Suppress HuggingFace and Python warnings
logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# Load object detection model
object_detector = pipeline(
    "object-detection",
    model="facebook/detr-resnet-50"
)

# Load image
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(image_url, stream=True).raw)

# Run detection
results = object_detector(image)

# Display only final results
for detection in results:
    print(f"{detection['label']} ({detection['score']:.2%}) at {detection['box']}")


###**Model 10: Image Captioning (Multimodal)**

In [ ]:
captioner = pipeline("image-to-text", model="Salesforce/blip-image-captioning-base")
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
image = Image.open(requests.get(image_url, stream=True).raw)
result = captioner(image)
print(f"Caption: {result[0]['generated_text']}")

In [ ]:
import os
import warnings
import logging
import requests
from PIL import Image

from transformers import AutoProcessor, BlipForConditionalGeneration
from transformers.utils import logging as tf_logging
from huggingface_hub.utils import logging as hub_logging

# --- Silenciar logs (mantener progress bar) ---
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"  # [web:152]
tf_logging.set_verbosity_error()                        # [web:152]
hub_logging.set_verbosity_error()                       # [web:152]
warnings.filterwarnings("ignore")
logging.getLogger("urllib3").setLevel(logging.ERROR)

# --- BLIP captioning (sin pipeline) ---
model_name = "Salesforce/blip-image-captioning-base"
processor = AutoProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name)

image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

inputs = processor(images=image, return_tensors="pt")
out_ids = model.generate(**inputs, max_new_tokens=30)

caption = processor.decode(out_ids[0], skip_special_tokens=True).strip()
print(f"Caption: {caption}")



##**THE 10 MOST POPULAR HUGGINFFACE MODELS TO RUN IN COLAB**
* Models with the highest download counts and most active community usage.
* Each one represents a category-defining breakthrough that's been battle-tested in production environments worldwide.

##**1. BERT Base Uncased (Google)**
* Model ID: bert-base-uncased
* Downloads: 50M+
* Parameters: 110M

>**What It Does:**  started the modern NLP revolution. **BERT (Bidirectional Encoder Representations from Transformers)** <mark>**understands context**</mark> from both directions in text, making it exceptional for classification, question answering, and named entity recognition.

> **Best Use:** Text classification, sentiment analysis, extracting information from documents, building search systems
> **Why It’s Popular:** 1st model to prove that pre-training on massive text corpora could dramatically improve performance across virtually every NLP task. Fast, reliable, and has been fine-tuned for thousands of specific applications.

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="bert-base-uncased")
result = classifier("This product exceeded all my expectations!")
print(result)

#### **EXPLANATION FOR WARNING & LABEL**

* **`bert-base-uncased`** is a **base BERT language model**, not trained for text classification.
* When you use it with `pipeline("text-classification")`, Hugging Face **adds a new classifier layer**.
* That layer is **randomly initialized**, so:

  * Labels like `LABEL_1` have **no real meaning**
  * The prediction is **not reliable**
* The ouput message is a **warning**, not an error.

> The warning appears because the base BERT model isn’t trained for classification; using a pretrained sentiment model gives meaningful labels and reliable results.

#### **Correct code** (use a trained classifier)
For sentiment (positive/negative) text like your example, use a **pretrained sentiment model**

####**let's fix the code with sentiment-analysis BERT classifier**





In [ ]:
from transformers import pipeline

# Load a pretrained sentiment analysis model
classifier = pipeline("sentiment-analysis")

text = (
    "They emailed saying the package will arrive in one week. "
    "This was supposed to arrive not later than tomorrow."
)

result = classifier(text)
print(result)


##**2. GPT-2 (OpenAI 2019)**
* Model ID: gpt2
* Downloads: 45M+
* Parameters: 124M (base), up to 1.5B (XL variant available)

**What It Does:** Generates coherent, contextually relevant text based on a prompt. The model that made AI-generated text mainstream.

**Best Use Cases:** Content generation, creative writing assistance, text completion, chatbots, code generation

In [ ]:
from transformers import pipeline
generator = pipeline("text-generation", model="gpt2")
result = generator("In the year 2030, artificial intelligence will", max_length=100, num_return_sequences=1)
print(result[0]['generated_text'])

#### **Warning output explain**

* The **download messages** show Hugging Face downloading GPT-2 and its tokenizer (this happens the first time only).
* The **warnings** explain configuration issues:

  * You set `max_length`, but truncation wasn’t explicitly enabled.
  * GPT-2 has no padding token, so it uses the **end-of-sequence token**.
  * Both `max_length` and `max_new_tokens` were set internally, causing a conflict.
* The **generated text** is a continuation of your prompt.
  It sounds repetitive and vague because **GPT-2 is an older (2019) model** and tends to ramble.

---
#### Corrected code (clean, no unnecessary warnings, output only)
* Uses `max_new_tokens` correctly
* Explicitly sets padding
* Suppresses non-critical warnings and logs
* Prints **only the generated text**

In [ ]:
import warnings
from transformers import pipeline, logging

# Suppress non-critical warnings and logs
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# Load text-generation model
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0
)

# Generate text
result = generator(
    "In the year 2030, artificial intelligence will",
    max_new_tokens=120,
    temperature=0.7,
    do_sample=True,
    pad_token_id=50256
)

# Display only the generated text
print(result[0]["generated_text"])


##<mark>**3. DistilBERT Base Uncased**</mark>
* Model ID: distilbert-base-uncased
* Downloads: 40M+
* Parameters: 66M

> **What It Does:** A **smaller, faster version of BERT** that **retains 97% of BERT’s language** understanding while being **60% faster + 40% smaller**.

> **Best Use Cases:** Real-time classification, mobile deployments, any BERT task where speed matters
* **`logging.set_verbosity_error()`** hides Hugging Face progress logs
* **`warnings.filterwarnings("ignore")`** suppresses non-critical warnings
* Real errors (e.g., model loading failures) will still be shown
* Output is **clean and assignment-ready**

*This is the **best practice setup** for demos and coursework.*
> **Why It’s Popular:** Perfect balance of performance + efficiency. BERT-quality results but can’t afford the computational cost, DistilBERT is the answer. It’s become the default choice for production deployments.


In [ ]:
import warnings
from transformers import pipeline, logging

# Suppress non-critical warnings and Hugging Face logs
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# Load sentiment analysis model
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Run sentiment analysis
result = sentiment("God, this rice bag if full of bugs")

# Display only the final output
print(f"Sentiment: {result[0]['label']}, Score: {result[0]['score']:.2%}")

### **4. RoBERTa Base (Facebook/Meta)**
* Model ID: roberta-base
* Downloads: 35M+
* Parameters: 125M

**What It Does:** An **optimized version of BERT trained** with improved methodology. RoBERTa (**Robustly Optimized BERT Approach**) often outperforms BERT on benchmarks.

**Best Use Cases:** When you need the absolute **best accuracy for classification, NER**, or question answering tasks

**Why It’s Popular:** RoBERTa consistently beats BERT in accuracy while maintaining similar speed. For mission-critical applications where accuracy is paramount, RoBERTa is the professional’s choice.

In [ ]:
from transformers import pipeline
qa = pipeline("question-answering", model="deepset/roberta-base-squad2")
context = "HuggingFace was founded in 2016 by Clément Delangue, Julien Chaumond, and Thomas Wolf. The company is headquartered in New York City."
question = "Where is HuggingFace headquartered?"
result = qa(question=question, context=context)
print(f"Answer: {result['answer']}")

###**5. T5 Base (Google)**
* Model ID: t5-base
* Downloads: 30M+
* Parameters: 220M (base), up to 11B (XXL variant)

**What It Does:** ***Text-to-Text Transfer*** Transformer. T5 treats **every NLP task as a text generation problem.** You give it an instruction and input, it gives you output.

**Best Use Cases:** ***Translation***, summarization, question answering, text classification (unified approach to multiple tasks)

**Why It’s Popular:** T5’s unified approach means you can use one model architecture for dozens of tasks. This simplifies deployment and maintenance. It’s Google’s workhorse model for production NLP systems.

In [ ]:
import os
import warnings
import logging

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as tf_logging
from huggingface_hub.utils import logging as hub_logging

# --- Silenciar logs (mantener progress bar) ---
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"  # quita avisos [web:59]
tf_logging.set_verbosity_error()
hub_logging.set_verbosity_error()
warnings.filterwarnings("ignore")
logging.getLogger("urllib3").setLevel(logging.ERROR)

model_name = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# --- Traducción EN -> DE con prefijo (T5 es “text-to-text”) ---
src = "HuggingFace makes AI accessible to everyone."
inputs = tokenizer("translate English to German: " + src, return_tensors="pt", truncation=True)
out_ids = model.generate(**inputs, max_new_tokens=60)
de = tokenizer.decode(out_ids[0], skip_special_tokens=True)
print(de)  # traducción

# --- Resumen con prefijo ---
long_text = (
    "Artificial intelligence is transforming every industry. Companies are using AI to automate repetitive tasks, "
    "gain insights from data, and create new products. The democratization of AI tools means that even small startups "
    "can now build sophisticated applications."
)
inputs = tokenizer("summarize: " + long_text, return_tensors="pt", truncation=True)
out_ids = model.generate(**inputs, max_new_tokens=30)
summary = tokenizer.decode(out_ids[0], skip_special_tokens=True)
print(summary)  # resumen


### **Mejor Modelo para text tranlation es Helsinki-NLP/Opus-mt**

In [ ]:
import os
import warnings
import logging
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as tf_logging
from huggingface_hub.utils import logging as hub_logging

# --- Silenciar logs (mantener progress bar) ---
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
tf_logging.set_verbosity_error()
hub_logging.set_verbosity_error()
warnings.filterwarnings("ignore")
logging.getLogger("urllib3").setLevel(logging.ERROR)

model_name = "Helsinki-NLP/opus-mt-tc-big-en-pt"  # EN -> PT [web:217]

tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPU + half precision si hay CUDA; si no, CPU en float32
use_cuda = torch.cuda.is_available()
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
)
if use_cuda:
    model = model.to("cuda")

english_text = (
    "Glaucoma is diagnosed using a combination of tests. Clinicians measure intraocular pressure, "
    "inspect the optic nerve for cupping, and perform visual field testing to detect peripheral vision loss. "
    "Optical coherence tomography (OCT) helps quantify retinal nerve fiber layer thinning, and gonioscopy "
    "assesses whether the drainage angle is open or closed."
)

inputs = tokenizer(english_text, return_tensors="pt", truncation=True)
if use_cuda:
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=140)

portuguese_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(portuguese_text)


###**6. CLIP (OpenAI)**
Model ID: openai/clip-vit-base-patch32
Downloads: 25M+
Parameters: 151M

**What It Does:** Connects images and text in the same embedding space. CLIP can match images to descriptions, classify images without training, and power image search systems.

**Best Use Cases:** Image search, zero-shot image classification, content moderation, matching products to descriptions

**Why It’s Popular:**CLIP revolutionized computer vision by eliminating the need for task-specific training. You can classify images into categories the model has never seen before. This makes it incredibly flexible for real-world applications where categories change frequently.

In [ ]:
import os
import warnings
import logging as py_logging

# Silenciar avisos generales
warnings.filterwarnings("ignore")

# Silenciar avisos "advisory" de Transformers (opcional)
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"  # [web:152]

# Silenciar logs de Hugging Face (NO es el logging estándar)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # [web:152]

# (Opcional) bajar ruido de requests/urllib3
py_logging.getLogger("urllib3").setLevel(py_logging.ERROR)

from transformers import pipeline
from PIL import Image
import requests

classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32"
)

image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

labels = ["a photo of a cat", "a photo of a dog", "a photo of a bird", "a photo of a car"]
results = classifier(image, candidate_labels=labels)

for r in results:
    print(f"{r['label']}: {r['score']:.2%}")


### **7. BART Large (Facebook/Meta)**
* Model ID: facebook/bart-large
* Downloads: 22M+
* Parameters: 406M

**What It Does:** Combines BERT’s bidirectional encoding with GPT’s autoregressive decoding. Exceptional at understanding and generating text, particularly for summarization and paraphrasing.

**Best Use Cases:** Summarization, paraphrasing, text simplification, zero-shot classification

**Why It’s Popular:** BART achieves state-of-the-art results on summarization tasks while remaining computationally efficient. News organizations and content platforms use BART to automatically generate article summaries.

In [ ]:
import os
import warnings
import logging

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import logging as hub_logging

# Silenciar logs (mantener progress bar)
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"  # [web:59]
hf_logging.set_verbosity_error()
hub_logging.set_verbosity_error()
warnings.filterwarnings("ignore")
logging.getLogger("urllib3").setLevel(logging.ERROR)

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

article = """
A deep learning convolutional neural network was trained to assess fundus
photographs and to predict SD OCT global RNFL thickness measurements.
The model was then tested on an independent sample of eyes with longitudinal
follow-up using both fundus photography and SD OCT. The ability to detect eyes
with statistically significant slopes of SD OCT change was assessed using
receiver operating characteristic (ROC) curves. The repeatability of RNFL
thickness predictions was evaluated using multiple photographs acquired
during the same day.
""".strip()

inputs = tokenizer(article, return_tensors="pt", truncation=True, max_length=1024)

summary_ids = model.generate(
    **inputs,
    max_new_tokens=50,   # equivalente a tu max_length=50 pero “solo lo nuevo” [web:248]
    min_new_tokens=20,
    num_beams=4,
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)


* Runs the **summarization model** on the text in **`article`**.
* Generates a summary that is **at least 20 tokens** long.
* Limits the summary to **no more than 50 tokens**.
* Stores the output (the summary text and metadata) in `result`.


###**8. Whisper Base (OpenAI)**

*   Model ID: openai/whisper-base
*   Downloads: 20M+
* Parameters: 74M

**What It Does:** <mark>***Automatic speech recognition (ASR)***</mark> that works across ***99 languages***. Whisper is remarkably robust to accents, background noise, and technical terminology.

**Best Use Cases**: Transcription services, voice assistants, accessibility features, meeting notes automation

**Why It’s Popular:** Whisper’s multilingual capabilities and accuracy in noisy conditions make it the default choice for speech recognition. Unlike previous ASR systems that required separate models per language, Whisper handles 99 languages with a single model.

In [ ]:
import os
import warnings
import logging as py_logging
from io import BytesIO
from math import gcd

import requests
import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly

from transformers import AutoProcessor, WhisperForConditionalGeneration
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import logging as hub_logging

# --- Silenciar logs (mantener progress bar) ---
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
hf_logging.set_verbosity_error()
hub_logging.set_verbosity_error()
warnings.filterwarnings("ignore")
py_logging.getLogger("urllib3").setLevel(py_logging.ERROR)

model_name = "openai/whisper-base"
processor = AutoProcessor.from_pretrained(model_name)  # [web:286]
model = WhisperForConditionalGeneration.from_pretrained(model_name)  # [web:286]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
resp = requests.get(audio_url)
resp.raise_for_status()

audio, sr = sf.read(BytesIO(resp.content), dtype="float32")
if audio.ndim == 2:
    audio = audio.mean(axis=1)

target_sr = 16000
if sr != target_sr:
    g = gcd(sr, target_sr)
    audio = resample_poly(audio, target_sr // g, sr // g).astype(np.float32)  # [web:300]
    sr = target_sr

inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    predicted_ids = model.generate(**inputs)

text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print(text)



In [ ]:
# --- Imports básicos (SO, warnings, logging, utilidades de bytes y math) ---
import os
import warnings
import logging as py_logging
from io import BytesIO
from math import gcd

# --- Imports para descargar y procesar audio (URL -> numpy, resample) ---
import requests
import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly  # resample eficiente a 16 kHz [web:300]

# --- Imports de Whisper (procesador + modelo) y logging HF para silenciar ruido ---
from transformers import AutoProcessor, WhisperForConditionalGeneration  # [web:286]
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import logging as hub_logging

# --- Silenciar logs (mantener progress bar) ---
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"  # desactiva “advisory warnings” [web:152]
hf_logging.set_verbosity_error()                       # baja verbosidad de Transformers [web:152]
hub_logging.set_verbosity_error()                      # baja verbosidad de huggingface_hub
warnings.filterwarnings("ignore")                      # silencia warnings de Python (opcional)
py_logging.getLogger("urllib3").setLevel(py_logging.ERROR)  # silencia logs de requests

# --- Configuración de modelo y dispositivo (A100 GPU) ---
model_name = "openai/whisper-base"
device = "cuda"

# --- Cargar processor y modelo Whisper en FP16 para acelerar en GPU ---
processor = AutoProcessor.from_pretrained(model_name)  # prepara features y decodifica texto [web:286]
model = WhisperForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,      # FP16 en GPU (rápido)
    low_cpu_mem_usage=True,         # menos RAM en carga
).to(device)

# --- Descargar audio de ejemplo (FLAC) desde una URL ---
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
resp = requests.get(audio_url)
resp.raise_for_status()

# --- Decodificar FLAC -> waveform (numpy) y convertir a mono si viene estéreo ---
audio, sr = sf.read(BytesIO(resp.content), dtype="float32")
if audio.ndim == 2:
    audio = audio.mean(axis=1)

# --- Re-muestrear a 16 kHz (Whisper espera 16000 Hz) ---
target_sr = 16000
if sr != target_sr:
    g = gcd(sr, target_sr)
    audio = resample_poly(audio, target_sr // g, sr // g).astype(np.float32)
    sr = target_sr

# --- Convertir waveform a input_features (tensor) y mover a GPU ---
inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# --- Inferencia: generar tokens de texto (sin gradientes + autocast FP16) ---
with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        predicted_ids = model.generate(**inputs)

# --- Decodificar tokens -> texto final y mostrar solo el resultado ---
text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]  # [web:286]
print(text)



###**9. Stable Diffusion v1.4 (CompVis/Stability AI)**

Model ID: CompVis/stable-diffusion-v1–4

Downloads: 18M+

Parameters: 860M

**What It Does:** Generates **high-quality images from text descriptions**. The model that **democratized AI art generation**.

**Best Use Cases:** Content creation, concept art, marketing materials, product visualization, creative exploration

**Why It’s Popular:** Stable Diffusion was the first high-quality text-to-image model released openly. Artists, designers, and marketers use it to rapidly prototype visual concepts. **Unlike DALL-E, it runs on consumer hardware**


In [ ]:
# Suppress non-critical warnings and logs
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

#=================================
# IMPORT STABLEDIFFUSION
#=================================
from diffusers import StableDiffusionPipeline
import torch

# Model checkpoint for Stable Diffusion
model_id = "CompVis/stable-diffusion-v1-4"

# Load the pretrained Stable Diffusion model using half precision for efficiency
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

# Move the model to the GPU for faster image generation
pipe = pipe.to("cuda")

# Text prompt describing the image to generate
prompt = (
    "a professional photograph of a modern AI research laboratory, "
    "cinematic lighting, highly detailed"
)

# Generate an image from the text prompt
image = pipe(prompt).images[0]

# Save the generated image to disk
image.save("generated_image.png")

# Confirm successful image generation
print("Image saved as generated_image.png")

In [ ]:
# Suppress non-critical warnings and logs
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

#=================================
# IMPORT STABLEDIFFUSION
#=================================
from diffusers import StableDiffusionPipeline
import torch

# Model checkpoint for Stable Diffusion
model_id = "CompVis/stable-diffusion-v1-4"

# Load the pretrained Stable Diffusion model using half precision for efficiency
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

# Move the model to the GPU for faster image generation
pipe = pipe.to("cuda")

# Text prompt describing the image to generate
prompt = (
    "a professional photograph of a modern AI research laboratory, "
    "cinematic lighting, highly detailed"
)

# Generate an image from the text prompt
image = pipe(prompt).images[0]

# Save the generated image to disk
image.save("generated_image.png")

# Confirm successful image generation
print("Image saved as generated_image.png")


**In short:**
Heavy Model, it can take some time depending on image. These lines show Stable Diffusion **downloading model files and assembling them** before generating images.:

* **model_index.json** – Main index that tells Diffusers which components the model has.
* **Fetching 16 files** – Downloading all required model parts.
* **text_encoder/model.safetensors** – Text encoder that converts your prompt into embeddings.
* **safety_checker/model.safetensors** – Checks generated images for unsafe content.
* **scheduler_config.json / scheduler_config-checkpoint.json** – Controls the diffusion steps (how noise is removed).
* **merges.txt / vocab.json / tokenizer_config.json / tokenizer.json** – Tokenizer files that process text prompts.
* **config.json (multiple)** – Configuration files for different model components.
* **preprocessor_config.json** – Image/text preprocessing settings.
* **unet/diffusion_pytorch_model.safetensors** – The core diffusion model (largest file).
* **vae/diffusion_pytorch_model.safetensors** – Converts latent images to final pixels.
* **special_tokens_map.json** – Special tokens used by the tokenizer.
* **Loading pipeline components…** – Assembling all downloaded parts into a working pipeline.

In [ ]:
from PIL import Image

# Load the saved image
img = Image.open("generated_image.png")

# Resize to max 400 px while keeping aspect ratio
img.thumbnail((400, 400))

# Display in Colab / Jupyter
display(img)


###**10. Vision Transformer (ViT) Base (Google)**
* Model ID: google/vit-base-patch16–224
* Downloads: 15M+
* Parameters: 86M

**What It Does:** Applies the transformer architecture (originally designed for text) to images. ViT treats images as sequences of patches, achieving state-of-the-art image classification.

**Best Use Cases:** Image classification, feature extraction for image search, transfer learning for custom vision tasks

**Why It’s Popular:** ViT proved that transformers aren’t just for text, they can match or exceed CNN performance on image tasks. This architectural unification has led to more efficient multimodal models. It’s become the foundation for modern computer vision systems.

In [ ]:
# Suppress non-critical warnings and logs
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# Import Hugging Face pipelines, image handling, and HTTP request utilities
from transformers import pipeline
from PIL import Image
import requests

# Load a Vision Transformer (ViT) model for image classification
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

# URL of the image to classify
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png"

# Download and open the image
image = Image.open(requests.get(image_url, stream=True).raw)

# Run image classification
results = classifier(image)

# Display the top 5 predicted labels with confidence scores
print("Top 5 predictions:")
for i, result in enumerate(results[:5], 1):
    print(f"{i}. {result['label']}: {result['score']:.2%}")


## **MY MODEL FROM HUGGINGFACE**
##**11. Facebook/wav2vec2-large-xlsr-53-spanish**

**Model ID:** `facebook/wav2vec2-large-xlsr-53-spanish`
**Downloads:** 5M+ (varies over time on Hugging Face)
**Parameters:** ~300M

**What It Does:**
Converts **spoken Spanish audio into text** using a self-supervised speech model. It is based on **Wav2Vec 2.0**, which learns speech representations directly from raw audio and is fine-tuned specifically for Spanish.

**Best Use Cases:**
* Spanish speech-to-text transcription
* Voice interfaces and assistants in Spanish
* Transcribing interviews, lectures, or podcasts
* Speech analysis and downstream NLP pipelines

**Why It’s Popular:**
It provides **high-quality Spanish ASR** without requiring Whisper, is **fully open source**, and works well on **free Colab GPUs**. The XLSR (cross-lingual speech representation) training allows strong performance even with limited labeled data, making it a go-to model for multilingual and Spanish speech recognition tasks.


**NOTE:** BY DEFECT this code takes .mp3 & .wav audio only.
> These libraries download audio, process it, and save it in the correct format.

* **`requests`**
  Downloads files or data from the web (e.g., fetch an audio file from a URL).

* **`librosa`**
  Loads and processes audio (resampling, converting to mono, extracting audio data).

* **`soundfile (sf)`**
  Saves audio data to files (e.g., write audio to a `.wav` file).



In [ ]:
import warnings
from transformers import pipeline, logging

# Suppress non-critical warnings and logs
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# Load Spanish speech-to-text model
asr = pipeline(
    "automatic-speech-recognition",
    model="facebook/wav2vec2-large-xlsr-53-spanish",
    device=0  # use GPU if available
)

# RAW audio file URL from your GitHub repository
audio_url = "https://raw.githubusercontent.com/GigiQR99/NLP-exercise/main/SPFemale_BlancaM-[AudioTrimmer.com].mp3"

# Transcribe audio directly from GitHub
result = asr(audio_url)

# Print transcription
print(result["text"])

## **ResNet-50 image classification** 📷
#### **Image + confidence + Top-5 bar chart**

* **What it does:** Classifies an image into **one main label** with a confidence score (Top-1, Top-5).
* **Architecture:** 50-layer CNN with **residual (skip) connections**.
* **Parameters:** ~25.6M
* **Training:** ImageNet (1,000 classes, ~1.2M images)
* **Model size:** ~98 MB (pretrained)

**Why I used**

* Strong, reliable **baseline**
* Fast inference, light
* Pretrained → works out of the box
* Good accuracy vs compute trade-off

**What it’s good for**

* Single-subject images
* Quick demos, teaching, benchmarking
* Getting **“what is in this image?”**

**Limitations**

* ❌ **Only one label per image**
* ❌ No bounding boxes or localization
* ❌ Struggles with multiple objects or cluttered scenes

**Bottom line**
> ResNet-50 tells you **what the main object is**, not **where objects are** or **how many there are**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, ResNetForImageClassification, logging
from PIL import Image
import requests
import torch
import matplotlib.pyplot as plt

logging.set_verbosity_error()

# Load image
image_url = "https://raw.githubusercontent.com/GigiQR99/CV-exercises/main/visuals/angelo-pantazis-wT3lf5qweEI-unsplash.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

# Load model
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
model.eval()

# Preprocess
inputs = processor(image, return_tensors="pt")

# Inference
with torch.no_grad():
    logits = model(**inputs).logits

# Decode prediction
predicted_class_id = logits.argmax(-1).item()
predicted_label = model.config.id2label[predicted_class_id]

print(f"Predicted object: {predicted_label}")

# Resize image (max 400px)
w, h = image.size
scale = 400 / max(w, h)
image_resized = image.resize((int(w * scale), int(h * scale)))

# Explicit render (THIS is the critical fix)
plt.imshow(image_resized)
plt.title(f"Predicted: {predicted_label}")
plt.axis("off")
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, ResNetForImageClassification, logging
from PIL import Image
import requests
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

logging.set_verbosity_error()

# Load image
url = "https://raw.githubusercontent.com/GigiQR99/CV-exercises/main/visuals/angelo-pantazis-wT3lf5qweEI-unsplash.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# Model
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
model.eval()

# Inference
inputs = processor(image, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits

probs = F.softmax(logits, dim=-1)[0]

# Top-1
top1_id = probs.argmax().item()
top1_label = model.config.id2label[top1_id]
top1_conf = probs[top1_id].item()

# Top-5 (convert to lists — CRITICAL FIX)
top5_probs, top5_ids = torch.topk(probs, 5)
top5_probs = top5_probs.cpu().tolist()
top5_labels = [model.config.id2label[i.item()] for i in top5_ids]

# Resize image (≤400 px)
w, h = image.size
scale = 400 / max(w, h)
image_resized = image.resize((int(w * scale), int(h * scale)))

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# Image panel
axes[0].imshow(image_resized)
axes[0].set_title(f"{top1_label} ({top1_conf:.2%})")
axes[0].axis("off")

# Bar chart panel (safe reversal)
axes[1].barh(top5_labels[::-1], top5_probs[::-1])
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Confidence")
axes[1].set_title("Top-5 Predictions")

plt.tight_layout()
plt.show()